# Premiers modèles de classification

## Prérequis

Cette
étape
repose
sur
la
finalisation
de
l’étape
2, au
cours
de
laquelle
les
données
ont
été
préparées
pour
la
modélisation.

Les
objets
suivants
sont
disponibles:

- `X`: matrice
de
features
entièrement
numérique, sans
valeurs
manquantes;
- `y`: variable
cible
binaire.

## Résultats attendus

L’objectif
est
de
construire:

- des
jeux
d’apprentissage
et
de
test;
- un
modèle
Dummy;
- un
modèle
linéaire;
- un
modèle
non - linéaire;

puis
d’évaluer
chacun
de
ces
modèles
sur
les
jeux
d’apprentissage
et
de
test
à
l’aide
de
métriques
de
classification
adaptées.

## 1. Import des bibliothèques

Cette étape charge les outils nécessaires à la séparation des données, à l’entraînement des modèles de classification et au calcul des métriques d’évaluation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option ("display.max_columns", 200)
sns.set_theme (style="whitegrid")

## 1. Chargement des données et reconstruction de `X` et `y`

Le
DataFrame
central
est
rechargé
depuis
le
dossier
`data / processed / `, puis
les
transformations
de
l’étape
2
sont
réappliquées
afin
d’obtenir
les
données
finales
de
modélisation.

In [ ]:
PROJECT_ROOT = Path.cwd ( ).resolve ( ).parent if Path.cwd ( ).name == "notebooks" else Path.cwd ( ).resolve ( )
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df_central = pd.read_csv (DATA_PROCESSED / "df_central.csv")

df_central["a_quitte_l_entreprise"] = (
    df_central["a_quitte_l_entreprise"]
    .astype (str)
    .str.strip ( )
    .str.capitalize ( )
)

df_central["attrition_bin"] = df_central["a_quitte_l_entreprise"].apply (
    lambda x: 1 if x == "Oui" else 0
)

print (df_central.shape)
display (df_central.head ( ))

In [ ]:
def split_features_by_type(df: pd.DataFrame):
    num_cols = df.select_dtypes (include=["number"]).columns.tolist ( )
    cat_cols = df.select_dtypes (include=["object", "string", "category"]).columns.tolist ( )
    return num_cols, cat_cols


def build_target(df: pd.DataFrame, target_col: str = "attrition_bin") -> pd.Series:
    return df[target_col].copy ( )


def build_feature_matrix(
        df: pd.DataFrame,
        cols_to_drop: list[str] | None = None
) -> pd.DataFrame:
    if cols_to_drop is None:
        cols_to_drop = [
            "a_quitte_l_entreprise",
            "attrition_bin",
            "id_employee",
            "code_sondage",
            "eval_number",
        ]
    return df.drop (columns=[col for col in cols_to_drop if col in df.columns]).copy ( )


def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy ( )
    num_cols, cat_cols = split_features_by_type (df)

    if not cat_cols:
        return df

    encoder = OneHotEncoder (
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    )

    encoded_array = encoder.fit_transform (df[cat_cols])

    encoded_df = pd.DataFrame (
        encoded_array,
        columns=encoder.get_feature_names_out (cat_cols),
        index=df.index
    )

    return pd.concat ([df[num_cols], encoded_df], axis=1)


def drop_highly_correlated_features(df: pd.DataFrame, threshold: float = 0.8):
    corr_matrix = df.corr (numeric_only=True).abs ( )
    upper = corr_matrix.where (np.triu (np.ones (corr_matrix.shape), k=1).astype (bool))
    to_drop = [column for column in upper.columns if any (upper[column] > threshold)]
    df_reduced = df.drop (columns=to_drop)
    return df_reduced, to_drop

In [ ]:
y = build_target (df_central)
X_raw = build_feature_matrix (df_central)
X_prepared = encode_features (X_raw)
X, dropped_corr_features = drop_highly_correlated_features (X_prepared, threshold=0.8)

print ("Shape final de X :", X.shape)
print ("Shape final de y :", y.shape)
print ("NaN dans X :", X.isna ( ).sum ( ).sum ( ))
print ("NaN dans y :", y.isna ( ).sum ( ))
print ("Variables supprimées pour forte corrélation :", dropped_corr_features)

## 2. Vérification finale des données de modélisation

La
matrice
`X`
et
la
cible
`y`
sont
maintenant
prêtes
pour
l’apprentissage
supervisé.

Les
contrôles
portent
sur:

- le
nombre
de
lignes;
- l’absence
de
valeurs
manquantes;
- la
compatibilité
des
données
avec
`scikit - learn`.

In [ ]:
assert len (X) == len (y), "X et y n'ont pas le même nombre de lignes"
assert X.isna ( ).sum ( ).sum ( ) == 0, "Il reste des NaN dans X"
assert y.isna ( ).sum ( ) == 0, "Il reste des NaN dans y"

print ("X et y sont prêts.")
print (y.value_counts ( ))

## 3. Séparation train / test

Les données sont séparées en un jeu d’apprentissage et un jeu de test, de manière stratifiée sur la cible afin de conserver une répartition comparable des classes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split (
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print ("X_train :", X_train.shape)
print ("X_test  :", X_test.shape)
print ("y_train :", y_train.shape)
print ("y_test  :", y_test.shape)

## 4. Définition des modèles

Trois
modèles
sont
retenus:

- un
modèle
Dummy
servant
de
benchmark;
- un
modèle
linéaire
de
type régression logistique;
- un
modèle
non - linéaire
de
type Random Forest.

In [ ]:
dummy_model = DummyClassifier (strategy="most_frequent", random_state=42)

linear_model = Pipeline (
    steps=[
        ("scaler", StandardScaler ( )),
        ("model", LogisticRegression (max_iter=3000, class_weight="balanced", random_state=42))
    ]
)

nonlinear_model = RandomForestClassifier (
    n_estimators=300,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

models = {
    "Dummy": dummy_model,
    "LogisticRegression": linear_model,
    "RandomForest": nonlinear_model,
}

In [ ]:
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit (X_train, y_train)

    y_pred_train = model.predict (X_train)
    y_pred_test = model.predict (X_test)

    rows = []
    for split_name, y_true, y_pred in [
        ("train", y_train, y_pred_train),
        ("test", y_test, y_pred_test),
    ]:
        rows.append ({
            "model": model_name,
            "split": split_name,
            "accuracy": accuracy_score (y_true, y_pred),
            "precision": precision_score (y_true, y_pred, zero_division=0),
            "recall": recall_score (y_true, y_pred, zero_division=0),
            "f1": f1_score (y_true, y_pred, zero_division=0),
        })

    return model, pd.DataFrame (rows), y_pred_train, y_pred_test

## 5. Entraînement et évaluation des modèles

Les trois modèles sont entraînés sur le jeu d’apprentissage, puis évalués sur les jeux d’apprentissage et de test afin de détecter un éventuel surapprentissage.

In [ ]:
trained_models = {}
results_list = {}
predictions = {}

for model_name, model in models.items ( ):
    trained_model, result_df, y_pred_train, y_pred_test = evaluate_model (
        model_name, model, X_train, X_test, y_train, y_test
    )
    trained_models[model_name] = trained_model
    results_list[model_name] = result_df
    predictions[model_name] = {
        "train": y_pred_train,
        "test": y_pred_test,
    }

results_df = pd.concat (results_list.values ( ), ignore_index=True).round (3)
display (results_df.sort_values (["split", "model"]))

## 6. Rapports de classification et matrices de confusion

Les métriques détaillées sont calculées pour chaque modèle sur les jeux d’apprentissage et de test.

Une attention particulière est portée à la classe positive, correspondant aux employés ayant quitté l’entreprise.

In [ ]:
for model_name in trained_models:
    print ("=" * 80)
    print (f"{model_name} - TRAIN")
    print ("=" * 80)
    print (classification_report (y_train, predictions[model_name]["train"], zero_division=0))

    print ("=" * 80)
    print (f"{model_name} - TEST")
    print ("=" * 80)
    print (classification_report (y_test, predictions[model_name]["test"], zero_division=0))

In [ ]:
for model_name in trained_models:
    fig, axes = plt.subplots (1, 2, figsize=(10, 4))

    ConfusionMatrixDisplay.from_predictions (
        y_train,
        predictions[model_name]["train"],
        ax=axes[0],
        cmap="Blues"
    )
    axes[0].set_title (f"{model_name} - Train")

    ConfusionMatrixDisplay.from_predictions (
        y_test,
        predictions[model_name]["test"],
        ax=axes[1],
        cmap="Blues"
    )
    axes[1].set_title (f"{model_name} - Test")

    plt.tight_layout ( )
    plt.show ( )

## 7. Interprétation des résultats

Le modèle Dummy obtient une accuracy élevée, mais un recall et une precision nuls sur la classe positive. Cela signifie qu’il ne détecte aucun employé quittant l’entreprise. Son rôle se limite donc à celui de benchmark.

La régression logistique présente le meilleur compromis sur le jeu de test. Son recall élevé montre qu’elle parvient à identifier une grande partie des employés susceptibles de quitter l’entreprise. Sa precision est plus modérée, ce qui traduit la présence d’un nombre non négligeable de faux positifs. Malgré cela, ce modèle apporte une réelle valeur ajoutée par rapport au Dummy.

Le modèle Random Forest obtient des performances quasi parfaites sur le jeu d’apprentissage, mais nettement plus faibles sur le jeu de test. Cet écart indique un surapprentissage important. En l’état, ce modèle généralise moins bien que la régression logistique.

À ce stade, la régression logistique constitue donc le modèle le plus pertinent pour la suite du projet, en particulier si l’objectif métier est de détecter le plus grand nombre possible de départs potentiels.

## 8. Vérification par rapport aux attendus de l’étape

Les attendus de cette étape sont remplis :

- les jeux `X_train`, `X_test`, `y_train` et `y_test` ont été construits ;
- trois modèles ont été entraînés : un modèle Dummy, un modèle linéaire et un modèle non-linéaire ;
- des métriques ont été calculées sur le jeu d’apprentissage et sur le jeu de test ;
- des matrices de confusion et des rapports de classification ont été produits pour chaque modèle.

L’analyse permet ainsi de comparer les performances des modèles, d’identifier la présence éventuelle de surapprentissage et d’évaluer leur capacité à détecter les employés quittant l’entreprise.

## 9. Interprétation des modèles

Le modèle Dummy sert de benchmark. Il obtient une accuracy élevée car il prédit essentiellement la classe majoritaire, mais il ne détecte aucun départ. Il n’apporte donc aucune valeur métier malgré son score global apparent.

La régression logistique constitue le meilleur compromis à ce stade. Elle présente un recall élevé sur la classe positive, ce qui signifie qu’elle détecte une part importante des employés susceptibles de quitter l’entreprise. Sa precision est plus modérée, ce qui traduit la présence de faux positifs, mais elle surpasse nettement le modèle Dummy.

La Random Forest présente des performances quasi parfaites sur le jeu d’apprentissage, mais nettement plus faibles sur le jeu de test. Cet écart met en évidence un surapprentissage. En l’état, elle généralise moins bien que la régression logistique.

Ces résultats montrent également qu’il ne faut pas confondre `accuracy` et `precision` : un modèle peut avoir une accuracy élevée tout en étant incapable de détecter correctement la classe minoritaire.

## Conclusion

Cette
étape
a
permis
de
construire
un
premier
benchmark
de
classification
conforme
aux
attendus:

- séparation
des
données
en
jeux
d’apprentissage
et
de
test;
- entraînement
d’un
modèle
Dummy, d’un
modèle
linéaire
et
d’un
modèle
non - linéaire;
- calcul
des
métriques
sur
train
et
test;
- analyse
de
l’overfitting
et
comparaison
avec
le
benchmark.

À
ce
stade, la
régression
logistique
apparaît
comme
le
modèle
le
plus
pertinent
pour
la
suite, car
elle
offre
le
meilleur
compromis
entre
capacité
de
généralisation
et
détection
des
départs.